# Stage 2 - Collect data

Pulls WildChat-1M, DevGPT, SWE-agent and OpenHands trajectories at the pinned versions in `configs/collect.yaml`, writes `raw/<name>.jsonl` and `raw/manifest.json` to Google Drive.

**Do a dry run first:** set each `limit` in `configs/collect.yaml` to about 200, run, check the counts, then raise the limits. DevGPT downloads a ~680 MB zip once.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[clean]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.collect.pipeline import run_collect

counts = run_collect("collect")            # all enabled datasets
# counts = run_collect("collect", only=["swe_agent"])   # one dataset
counts

In [ ]:
import pandas as pd
from krs.collect import manifest

pd.DataFrame(manifest.read()).T   # source, version, licence, rows, date per dataset

In [ ]:
# Quick look at one record per dataset
from krs.config import get_paths
from krs.io import read_jsonl
from krs.schema import Conversation

for name in counts:
    c = next(read_jsonl(get_paths().raw / f"{name}.jsonl", Conversation))
    print(name, '|', c.conversation_id, '| project:', c.project_id, '| turns:', len(c.turns))
    print('  ', c.turns[1].content[:200].replace('\n', ' ') if len(c.turns) > 1 else '')

## Own logs - not on Colab

Consented team logs are collected **locally only**, never on Colab or Drive, and must be redacted (Stage 3) before anything else uses them:

```python
from krs.collect.pipeline import run_own_logs
run_own_logs("own_logs/consent.yaml")   # template: configs/consent.example.yaml
```